# Chapter 5: Risk-averse models

## Setup

In [1]:
import os

import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

os.makedirs("results", exist_ok=True)

k, s, r = 15, 45, 10
GROUPS = [[0, 1, 2], [3, 4, 5], [6, 7, 8, 9]]
Q_COLS = [f"Q{i}" for i in range(1, 11)]
ALPHAS = [0.01, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1]
N, SEED, TIME_LIMIT = 1000, 8000, 120
# the objective is bounded, so INF_OR_UNBD means infeasible
STATUS = {GRB.OPTIMAL: "optimal", GRB.INFEASIBLE: "infeasible", GRB.INF_OR_UNBD: "infeasible",
          GRB.TIME_LIMIT: "time limit"}

p_bench = np.array([0.05, 0.25, 0.15, 0.10, 0.20, 0.08, 0.12, 0.04, 0.17, 0.22])
y_bench = np.array([0.80, 0.40, 0.50, 0.70, 0.40, 0.60, 0.75, 0.90, 0.65, 0.85])
c_bench = np.array([40, 25, 35, 30, 21, 55, 45, 37, 52, 42], dtype=float)

In [2]:
# instance l* = 3, identical to chapter 4
np.random.seed(3003)
p = np.random.uniform(np.maximum(0.02, p_bench - 0.05), np.minimum(0.35, p_bench + 0.05))
y = np.random.uniform(np.maximum(0.30, y_bench - 0.15), np.minimum(0.95, y_bench + 0.15))
c = np.random.uniform(np.maximum(r + 1, 0.90 * c_bench), np.minimum(s + k - 1, 1.10 * c_bench))
mu = np.random.uniform(350, 450)
sigma = np.random.uniform(0.20, 0.29) * mu
inst = {"p": p, "y": y, "c": c, "mu": mu, "sigma": sigma}

print(f"mu = {mu:.2f}, sigma = {sigma:.2f}")
pd.DataFrame({"supplier": range(1, 11), "p": p, "y": y, "c": c}).round(3)

mu = 353.12, sigma = 76.03


,supplier,p,y,c
0,1,0.047,0.713,38.558
1,2,0.271,0.364,24.806
2,3,0.111,0.575,34.965
3,4,0.067,0.721,31.365
4,5,0.204,0.426,20.145
5,6,0.105,0.671,50.144
6,7,0.110,0.666,47.579
7,8,0.045,0.869,39.397
8,9,0.161,0.708,51.580
9,10,0.186,0.912,43.592


## Functions

In [3]:
# identical to chapter 4
def generate(N, seed, inst, E_fixed=None):
    rng = np.random.default_rng(seed)
    n = len(inst["p"])
    # group events and number of hit groups
    E = (rng.random((N, 3)) < 0.15).astype(int)
    if E_fixed is not None:
        E = np.tile(np.asarray(E_fixed, dtype=int), (N, 1))
    H = E.sum(axis=1)
    # disruption probabilities and yield factors
    p_mod = np.tile(inst["p"], (N, 1))
    for g, members in enumerate(GROUPS):
        hit = E[:, g] == 1
        p_mod[np.ix_(hit, members)] = np.minimum(0.90, inst["p"][members] + 0.25)
    R = np.where(rng.random((N, n)) < p_mod, inst["y"], 1.0)
    # stressed demand and spot price
    D = np.maximum(0.0, (1 + 0.1 * H) * rng.normal(inst["mu"], inst["sigma"], N))
    SP = 30 + 15 * H + 30 * rng.random(N)
    return H, R, D, SP


def profit(Q, R, D, SP, inst):
    X = R @ Q
    shortage = np.maximum(0.0, D - X)
    # each unit short is bought on the spot market or lost, whichever is cheaper
    return (s - r) * D + r * X - R @ (inst["c"] * Q) - np.minimum(s - r + k, SP - r) * shortage


def active(Q):
    return " ".join(str(i + 1) for i in np.flatnonzero(Q > 1e-6))

In [4]:
def add_recourse(m, R, D, SP, inst):
    N, n = R.shape
    # first stage: order quantities
    Q = m.addMVar(n, lb=0.0)
    # second stage: sales, leftover, spot purchase, lost sales
    z = m.addMVar(N, lb=0.0)
    v = m.addMVar(N, lb=0.0)
    B = m.addMVar(N, lb=0.0)
    L = m.addMVar(N, lb=0.0)
    # supply + spot = sales + leftover; sales + lost sales = demand (as in the SAA LP of chapter 4)
    m.addConstr(z + v - R @ Q - B == 0)
    m.addConstr(z + L == D)
    # profit per scenario
    return Q, s * z + r * v - SP * B - k * L - (R * inst["c"]) @ Q


def risk(Q, alpha, R, D, SP, inst):
    Pi = np.sort(profit(Q, R, D, SP, inst))
    n_tail = round(alpha * len(Pi))
    # VaR = (alpha N + 1)-th smallest profit (none at alpha = 1), CVaR = mean of the alpha N worst
    VaR = Pi[n_tail] if n_tail < len(Pi) else np.nan
    return {"mean": Pi.mean(), "sd": Pi.std(ddof=1), "VaR": VaR, "CVaR": Pi[:n_tail].mean()}

In [5]:
def solve_rn(R, D, SP, inst):
    with gp.Model(env=env) as m:
        Q, Pi = add_recourse(m, R, D, SP, inst)
        # expected profit
        m.setObjective(Pi.sum() / len(D), GRB.MAXIMIZE)
        m.optimize()
        return Q.X.copy(), m.ObjVal


def solve_cvar(alpha, R, D, SP, inst):
    with gp.Model(env=env) as m:
        Q, Pi = add_recourse(m, R, D, SP, inst)
        # shortfall eta_w of the profit below t
        t = m.addVar(lb=-GRB.INFINITY)
        eta = m.addMVar(len(D), lb=0.0)
        m.addConstr(eta >= t - Pi)
        # CVaR: t minus the average shortfall over the alpha-tail
        m.setObjective(t - eta.sum() / (alpha * len(D)), GRB.MAXIMIZE)
        m.optimize()
        return Q.X.copy(), m.ObjVal


def solve_var(alpha, tau, M, R, D, SP, inst, Q_start=None):
    with gp.Model(env=env) as m:
        Q, Pi = add_recourse(m, R, D, SP, inst)
        # valid inequality: orders with y @ Q > D_max are dominated in every scenario
        m.addConstr(inst["y"] @ Q <= D.max())
        if alpha < 1:
            # at most alpha N scenarios below tau (vacuous at alpha = 1)
            phi = m.addMVar(len(D), vtype=GRB.BINARY)
            m.addConstr(Pi >= tau - M * phi)
            m.addConstr(phi.sum() <= alpha * len(D))
        if Q_start is not None:
            Q.Start = Q_start
        m.setObjective(Pi.sum() / len(D), GRB.MAXIMIZE)
        m.Params.TimeLimit = TIME_LIMIT
        m.optimize()
        status = STATUS.get(m.Status, str(m.Status))
        if m.SolCount == 0:
            return None, status, np.nan, m.Runtime
        return Q.X.copy(), status, m.MIPGap if m.IsMIP else 0.0, m.Runtime

## Risk-neutral reference

In [6]:
# one fixed scenario set for all models
_, R, D, SP = generate(N, SEED, inst)

# risk-neutral solution, with VaR and CVaR per alpha
Q_RN, z_RN = solve_rn(R, D, SP, inst)
rn = pd.DataFrame([{"alpha": alpha, "active": active(Q_RN), **dict(zip(Q_COLS, Q_RN)),
                    **risk(Q_RN, alpha, R, D, SP, inst)} for alpha in ALPHAS])
rn.round(2)

,alpha,active,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,mean,sd,VaR,CVaR
0,0.01,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,249.40,-11.15
1,0.10,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,2832.61,1607.11
2,0.20,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,4179.67,2584.76
3,0.40,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,6179.68,3914.15
4,0.60,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,8090.46,5020.55
5,0.80,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,10076.77,6027.07
6,0.90,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,10933.32,6528.62
7,1.00,5,0.0,0.0,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,NaN,6997.13


## 5.1 CVaR model

In [7]:
Q_cvar, obj, rows = {}, {}, []
for alpha in ALPHAS:
    Q_cvar[alpha], obj[alpha] = solve_cvar(alpha, R, D, SP, inst)
    rows.append({"alpha": alpha, "active": active(Q_cvar[alpha]), **dict(zip(Q_COLS, Q_cvar[alpha])),
                 **risk(Q_cvar[alpha], alpha, R, D, SP, inst)})
cvar = pd.DataFrame(rows)

# LP objective against the CVaR of the realized profits, and alpha = 1 against risk-neutral
print(f"max |LP objective - CVaR| = {np.abs(cvar['CVaR'] - pd.Series(obj.values())).max():.1e}")
print(f"alpha = 1: |objective - z_RN| = {abs(obj[1] - z_RN):.1e}, max |Q - Q_RN| = {np.abs(Q_cvar[1] - Q_RN).max():.1e}")
cvar.round(2)

max |LP objective - CVaR| = 2.6e-11
alpha = 1: |objective - z_RN| = 9.5e-11, max |Q - Q_RN| = 0.0e+00


,alpha,active,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,mean,sd,VaR,CVaR
0,0.01,2 5,0.0,19.72,0.0,0.0,476.72,0.0,0.0,0.0,0.0,0.0,6926.83,2833.51,860.11,447.07
1,0.10,2 5,0.0,59.53,0.0,0.0,473.83,0.0,0.0,0.0,0.0,0.0,6794.69,2641.84,3577.88,2351.91
2,0.20,2 5,0.0,78.49,0.0,0.0,446.86,0.0,0.0,0.0,0.0,0.0,6814.47,2622.98,4588.21,3249.60
3,0.40,2 5,0.0,156.77,0.0,0.0,323.53,0.0,0.0,0.0,0.0,0.0,6750.88,2446.84,6264.08,4339.91
4,0.60,2 5,0.0,140.03,0.0,0.0,324.21,0.0,0.0,0.0,0.0,0.0,6774.70,2463.66,7614.95,5175.84
5,0.80,2 5,0.0,8.53,0.0,0.0,413.81,0.0,0.0,0.0,0.0,0.0,6925.22,2938.02,9867.49,6080.59
6,0.90,5,0.0,0.00,0.0,0.0,437.78,0.0,0.0,0.0,0.0,0.0,6982.40,2991.11,10664.57,6547.13
7,1.00,5,0.0,0.00,0.0,0.0,454.89,0.0,0.0,0.0,0.0,0.0,6997.13,2982.33,NaN,6997.13


## 5.2 VaR-constrained model

In [8]:
# smallest valid big-M: lowest profit per scenario over the vertices of {Q >= 0, y @ Q <= D_max}
vertices = [np.zeros(10)] + [D.max() / y[i] * np.eye(10)[i] for i in range(10)]
Pi_low = np.min([profit(V, R, D, SP, inst) for V in vertices], axis=0)

rows = []
for alpha in ALPHAS:
    # threshold: VaR of the CVaR solution, which is also the MIP start
    tau = risk(Q_cvar[alpha], alpha, R, D, SP, inst)["VaR"]
    M = np.maximum(0.0, tau - Pi_low)
    Q_hat, status, gap, runtime = solve_var(alpha, tau, M, R, D, SP, inst, Q_start=Q_cvar[alpha])
    if Q_hat is None:
        rows.append({"alpha": alpha, "tau": tau, "status": status, "runtime_s": runtime})
        continue
    # scenarios below tau, with a small tolerance for scenarios at tau
    n_below = (profit(Q_hat, R, D, SP, inst) < tau - 1e-6).sum()
    rows.append({"alpha": alpha, "tau": tau, "status": status, "active": active(Q_hat), **dict(zip(Q_COLS, Q_hat)),
                 **risk(Q_hat, alpha, R, D, SP, inst), "n_below": n_below, "mip_gap": gap, "runtime_s": runtime})
var = pd.DataFrame(rows)

print("n_below <= alpha N in every row:", bool((var["n_below"] <= var["alpha"] * N).all()))
print(f"alpha = 1: |mean - z_RN| = {abs(var['mean'].iloc[-1] - z_RN):.1e}, "
      f"max |Q - Q_RN| = {np.abs(var[Q_COLS].iloc[-1].to_numpy(dtype=float) - Q_RN).max():.1e}")
var.round(2).assign(mip_gap=var["mip_gap"].round(6))

n_below <= alpha N in every row: True
alpha = 1: |mean - z_RN| = 1.2e-10, max |Q - Q_RN| = 0.0e+00


,alpha,tau,status,active,Q1,Q2,Q3,Q4,Q5,Q6,...,Q8,Q9,Q10,mean,sd,VaR,CVaR,n_below,mip_gap,runtime_s
0,0.01,860.11,optimal,2 5,0.0,20.30,0.0,0.0,475.51,0.0,...,0.0,0.0,0.0,6928.31,2833.24,860.11,445.87,10,0.000000,1.75
1,0.10,3577.88,time limit,2 5,0.0,91.24,0.0,0.0,400.20,0.0,...,0.0,0.0,0.0,6883.60,2625.92,3577.88,2270.71,100,0.007161,120.10
2,0.20,4588.21,time limit,2 5,0.0,70.50,0.0,0.0,411.34,0.0,...,0.0,0.0,0.0,6921.03,2694.97,4588.21,3127.00,200,0.006241,120.04
3,0.40,6264.08,optimal,5,0.0,0.00,0.0,0.0,445.52,0.0,...,0.0,0.0,0.0,6993.34,2989.95,6264.08,3883.58,400,0.000000,2.54
4,0.60,7614.95,optimal,5,0.0,0.00,0.0,0.0,454.89,0.0,...,0.0,0.0,0.0,6997.13,2982.33,8090.46,5020.55,542,0.000000,0.34
5,0.80,9867.49,optimal,5,0.0,0.00,0.0,0.0,454.89,0.0,...,0.0,0.0,0.0,6997.13,2982.33,10076.77,6027.07,781,0.000000,0.48
6,0.90,10664.57,optimal,5,0.0,0.00,0.0,0.0,454.89,0.0,...,0.0,0.0,0.0,6997.13,2982.33,10933.32,6528.62,861,0.000000,0.50
7,1.00,NaN,optimal,5,0.0,0.00,0.0,0.0,454.89,0.0,...,0.0,0.0,0.0,6997.13,2982.33,NaN,6997.13,0,0.000000,0.01


In [9]:
# feasibility check: fixed threshold tau = 0.5 z_RN, without MIP start
tau_fix = 0.5 * z_RN
print(f"tau = {tau_fix:.2f}")
for alpha in [0.01, 0.1]:
    Q_hat, status, _, runtime = solve_var(alpha, tau_fix, np.maximum(0.0, tau_fix - Pi_low), R, D, SP, inst)
    n_below = "-" if Q_hat is None else (profit(Q_hat, R, D, SP, inst) < tau_fix - 1e-6).sum()
    print(f"alpha = {alpha}: {status if Q_hat is None else 'feasible'}, scenarios below tau = {n_below}, "
          f"runtime = {runtime:.1f} s")

tau = 3498.57


alpha = 0.01: infeasible, scenarios below tau = -, runtime = 1.3 s


alpha = 0.1: feasible, scenarios below tau = 100, runtime = 14.1 s


## Export

In [10]:
rn.round(2).to_csv("results/5_0_risk_neutral.csv", index=False, na_rep="-")
cvar.round(2).to_csv("results/5_1_cvar.csv", index=False, na_rep="-")
var.round(2).assign(mip_gap=var["mip_gap"].round(6)).to_csv("results/5_2_var.csv", index=False, na_rep="-")